# Fine-tuned TabPFN: combined results

Merges the per-layer outputs of the four separate fine-tuning notebooks into one place:

1. the four `fm_subsampled_summary_layer_*.csv` files into a single summary table
2. the checkpoint folders (`layer_*_subsampled/`) into an inventory table (best/interval checkpoints, epoch, size, saved metrics)
3. a cross-check that the two sources agree on the best epoch

In [1]:
from pathlib import Path
import re

import numpy as np
import pandas as pd

try:
    import torch
except ImportError:
    torch = None

import sys
scripts_dir = (Path.cwd() / ".." / "scripts").resolve()
sys.path.append(str(scripts_dir))
from config import final_data_path

final_data_path = Path(final_data_path)
features_dir = final_data_path
results_dir = final_data_path / "modeling_results" / "fm_tuned"
tuned_dir = results_dir
results_dir.mkdir(parents=True, exist_ok=True)

layer_files = [
    "layer_a_bts",
    "layer_b_bts_meteostat",
    "layer_c_bts_gdelt",
    "layer_d_bts_meteostat_gdelt",
]
layer_keys = [f"{name}_subsampled" for name in layer_files]
layer_labels = dict(zip(layer_keys, ["A: BTS", "B: BTS+Weather", "C: BTS+GDELT", "D: BTS+Weather+GDELT"]))

print("tuned_dir exists:", tuned_dir.exists())


tuned_dir exists: True


## 1. Combine the per-layer summary files

In [2]:
summary_paths = sorted(tuned_dir.glob("fm_subsampled_summary_layer_*.csv"))
print(f"found {len(summary_paths)} summary files:")
for p in summary_paths:
    print("  -", p.name)

frames = []
for p in summary_paths:
    df = pd.read_csv(p)
    df["source_file"] = p.name
    frames.append(df)

fm_ft_df = pd.concat(frames, ignore_index=True)

# a layer appearing twice means a rerun left two files -- keep the last one read, but say so
dupes = fm_ft_df["layer"].duplicated(keep=False)
if dupes.any():
    print("WARNING: layers present more than once, keeping the last occurrence:")
    print(fm_ft_df.loc[dupes, ["layer", "source_file"]].to_string(index=False))
    fm_ft_df = fm_ft_df.drop_duplicates(subset="layer", keep="last")

missing = [k for k in layer_keys if k not in set(fm_ft_df["layer"])]
if missing:
    print("WARNING: no summary found for:", missing)

fm_ft_df["layer"] = pd.Categorical(fm_ft_df["layer"], categories=layer_keys, ordered=True)
fm_ft_df = fm_ft_df.sort_values("layer").reset_index(drop=True)
fm_ft_df.insert(1, "layer_label", fm_ft_df["layer"].map(layer_labels))

# all four layers should have been run with the same fine-tuning setup
for col in ["learning_rate", "epochs_cap", "n_estimators_finetune", "n_estimators_final_inference"]:
    if col in fm_ft_df.columns and fm_ft_df[col].nunique() > 1:
        print(f"WARNING: {col} differs between layers: {fm_ft_df[col].tolist()}")

fm_ft_df.to_csv(results_dir / "fm_finetuned_summary.csv", index=False)
fm_ft_df


found 4 summary files:
  - fm_subsampled_summary_layer_a.csv
  - fm_subsampled_summary_layer_b.csv
  - fm_subsampled_summary_layer_c.csv
  - fm_subsampled_summary_layer_d.csv


,layer,layer_label,f1,precision,recall,auc_roc,auc_pr,mcc,threshold,learning_rate,epochs_cap,epochs_run,best_epoch,best_val_auc,n_estimators_finetune,n_estimators_final_inference,fit_time_s,source_file
0,layer_a_bts_subsampled,A: BTS,0.423377,0.312681,0.655406,0.673820,0.357110,0.209519,0.184379,0.00003,20,20,17,0.710571,2,2,32192.347818,fm_subsampled_summary_layer_a.csv
1,layer_b_bts_meteostat_subsampled,B: BTS+Weather,0.431440,0.316857,0.675837,0.684994,0.366396,0.222382,0.183544,0.00003,20,20,19,0.719091,2,2,30801.036381,fm_subsampled_summary_layer_b.csv
2,layer_c_bts_gdelt_subsampled,C: BTS+GDELT,0.419958,0.319439,0.612787,0.671538,0.349539,0.207512,0.202246,0.00003,20,20,17,0.714251,2,2,36285.774945,fm_subsampled_summary_layer_c.csv
3,layer_d_bts_meteostat_gdelt_subsampled,D: BTS+Weather+GDELT,0.429590,0.324072,0.636997,0.682408,0.360264,0.221674,0.195745,0.00003,20,20,19,0.721518,2,2,32830.457027,fm_subsampled_summary_layer_d.csv


## 2. Checkpoint inventory

`fit()` writes two kinds of `.pth` files into each layer's output folder: `checkpoint_<size>_best.pth` (overwritten whenever validation AUC improves) and interval checkpoints `checkpoint_<size>_<epoch>.pth` (every 10 epochs by default). Each stores its epoch and the validation metrics at that point, which is read below without using the model weights.

In [3]:
def read_checkpoint_meta(path):
    """Numeric fields stored next to the weights (epoch, validation metrics). Empty if torch is unavailable."""
    if torch is None:
        return {}
    try:
        ckpt = torch.load(path, map_location="cpu", weights_only=False)
    except Exception as e:
        return {"read_error": type(e).__name__}
    meta = {}
    if isinstance(ckpt, dict):
        for k, v in ckpt.items():
            if isinstance(v, (int, float)) and not isinstance(v, bool):
                meta[k] = float(v)
    del ckpt
    return meta


rows = []
for key in layer_keys:
    layer_dirs = [d for d in tuned_dir.rglob(key) if d.is_dir()]
    if not layer_dirs:
        print(f"no checkpoint folder found for {key}")
        continue
    for layer_dir in layer_dirs:
        for f in sorted(layer_dir.glob("*.pth")):
            m = re.search(r"_(best|\d+)\.pth$", f.name)
            kind = m.group(1) if m else "unknown"
            row = {
                "layer": key,
                "kind": "best" if kind == "best" else "interval",
                "epoch_in_name": int(kind) if kind.isdigit() else np.nan,
                "file": f.name,
                "size_mb": round(f.stat().st_size / 1e6, 1),
                "modified": pd.to_datetime(f.stat().st_mtime, unit="s"),
                "folder": str(layer_dir),
            }
            row.update(read_checkpoint_meta(f))
            rows.append(row)

ckpt_df = pd.DataFrame(rows)
if ckpt_df.empty:
    print("no .pth files found -- check that the checkpoint folders were uploaded with the dataset "
          "and that fit() was called with output_dir")
else:
    ckpt_df["layer"] = pd.Categorical(ckpt_df["layer"], categories=layer_keys, ordered=True)
    ckpt_df = ckpt_df.sort_values(["layer", "kind", "epoch_in_name"]).reset_index(drop=True)
    ckpt_df.to_csv(results_dir / "fm_finetuned_checkpoints_all_layers.csv", index=False)
ckpt_df


,layer,kind,epoch_in_name,file,size_mb,modified,folder,epoch,primary_metric,log_loss,roc_auc
0,layer_a_bts_subsampled,best,NaN,checkpoint_850K_best.pth,2554.0,2026-10-04 18:19:08,/Users/aniaandruszkiewicz/FS/Thesis/data/final...,18.0,0.710571,0.465054,0.710571
1,layer_a_bts_subsampled,interval,10.0,checkpoint_850K_10.pth,2554.0,2026-10-04 17:52:08,/Users/aniaandruszkiewicz/FS/Thesis/data/final...,10.0,0.707539,0.466567,0.707539
2,layer_a_bts_subsampled,interval,20.0,checkpoint_850K_20.pth,2554.0,2026-10-04 17:59:36,/Users/aniaandruszkiewicz/FS/Thesis/data/final...,20.0,0.710560,0.465025,0.710560
3,layer_b_bts_meteostat_subsampled,best,NaN,checkpoint_850K_best.pth,2554.0,2026-10-04 18:24:42,/Users/aniaandruszkiewicz/FS/Thesis/data/final...,19.0,0.719086,0.460079,0.719086
4,layer_b_bts_meteostat_subsampled,interval,10.0,checkpoint_850K_10.pth,2554.0,2026-10-04 18:01:46,/Users/aniaandruszkiewicz/FS/Thesis/data/final...,10.0,0.715197,0.462428,0.715197
5,layer_b_bts_meteostat_subsampled,interval,20.0,checkpoint_850K_20.pth,2554.0,2026-10-04 18:13:16,/Users/aniaandruszkiewicz/FS/Thesis/data/final...,20.0,0.719091,0.460080,0.719091
6,layer_c_bts_gdelt_subsampled,best,NaN,checkpoint_850K_best.pth,2554.0,2026-10-05 04:06:18,/Users/aniaandruszkiewicz/FS/Thesis/data/final...,18.0,0.714251,0.463013,0.714251
7,layer_c_bts_gdelt_subsampled,interval,10.0,checkpoint_850K_10.pth,2554.0,2026-10-05 03:36:34,/Users/aniaandruszkiewicz/FS/Thesis/data/final...,10.0,0.709307,0.465733,0.709307
8,layer_c_bts_gdelt_subsampled,interval,20.0,checkpoint_850K_20.pth,2554.0,2026-10-05 03:51:20,/Users/aniaandruszkiewicz/FS/Thesis/data/final...,20.0,0.714231,0.462928,0.714231
9,layer_d_bts_meteostat_gdelt_subsampled,best,NaN,checkpoint_850K_best.pth,2554.0,2026-10-05 04:06:44,/Users/aniaandruszkiewicz/FS/Thesis/data/final...,18.0,0.721514,0.458584,0.721514


In [4]:
# one row per layer: the best checkpoint
if not ckpt_df.empty:
    best_ckpt_df = ckpt_df[ckpt_df["kind"] == "best"].drop(columns=["kind", "epoch_in_name", "folder"])
    best_ckpt_df.to_csv(results_dir / "fm_finetuned_best_checkpoints.csv", index=False)
else:
    best_ckpt_df = pd.DataFrame()
best_ckpt_df


,layer,file,size_mb,modified,epoch,primary_metric,log_loss,roc_auc
0,layer_a_bts_subsampled,checkpoint_850K_best.pth,2554.0,2026-10-04 18:19:08,18.0,0.710571,0.465054,0.710571
3,layer_b_bts_meteostat_subsampled,checkpoint_850K_best.pth,2554.0,2026-10-04 18:24:42,19.0,0.719086,0.460079,0.719086
6,layer_c_bts_gdelt_subsampled,checkpoint_850K_best.pth,2554.0,2026-10-05 04:06:18,18.0,0.714251,0.463013,0.714251
9,layer_d_bts_meteostat_gdelt_subsampled,checkpoint_850K_best.pth,2554.0,2026-10-05 04:06:44,18.0,0.721514,0.458584,0.721514


## 3. Cross-check: summary vs. checkpoints

The summary's `best_epoch` comes from the training logger, which counts epochs from 0. The checkpoint stores `epoch + 1`, so the two are expected to differ by exactly 1. A mismatch means the files come from different runs.

In [5]:
if not best_ckpt_df.empty and "epoch" in best_ckpt_df.columns:
    check_df = fm_ft_df[["layer", "best_epoch", "epochs_run", "epochs_cap"]].merge(
        best_ckpt_df[["layer", "epoch"]].rename(columns={"epoch": "checkpoint_best_epoch"}),
        on="layer", how="left",
    )
    check_df["consistent"] = check_df["checkpoint_best_epoch"] == check_df["best_epoch"] + 1
    check_df["hit_epoch_cap"] = check_df["epochs_run"] >= check_df["epochs_cap"]
else:
    check_df = pd.DataFrame()
    print("cross-check skipped (no checkpoints read)")
check_df


,layer,best_epoch,epochs_run,epochs_cap,checkpoint_best_epoch,consistent,hit_epoch_cap
0,layer_a_bts_subsampled,17,20,20,18.0,True,True
1,layer_b_bts_meteostat_subsampled,19,20,20,19.0,False,True
2,layer_c_bts_gdelt_subsampled,17,20,20,18.0,True,True
3,layer_d_bts_meteostat_gdelt_subsampled,19,20,20,18.0,False,True
